# 🗓️ 33일차 스터디 노트북 — 이진 검색 트리 (BST)

**오늘 범위**: 09-2 이진 트리와 이진 검색 트리 — 이진 트리 · 완전 이진 트리 · BST 조건 · `search` · `add` · `remove`(3경우) · `dump` · `min_key`/`max_key` · 보충수업 9-1(균형 검색 트리), 9-2(내림차순 덤프)

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[그림]** 트리 직접 그리기 · **[손]** 손으로 추적 · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]**

---

## 어제 배운 게 전부 코드로 나온다

32일차는 100% 개념이었지. 오늘 그게 하나씩 코드로 나타나:

| 32일차 개념 | 오늘 어디에 |
|---|---|
| **중위 순회** | `dump()` — 오름차순 출력의 정체 |
| **차수**(자식 수) | `remove()` 가 **0개/1개/2개** 세 경우로 갈리는 이유 |
| **높이** | 검색 성능을 좌우 (실측 10,000번 vs 6번!) |
| **완전 이진 트리** | 24일차 힙 정렬에서 배열에 담았던 그 구조 |
| **빈 트리** | `None` — 재귀의 종료 조건 |

## 🎯 오늘의 진행 방식

> **손 → 손 → 손 → 코드**

**검색·삽입·삭제를 전부 손으로 그려본 뒤에** 코드로 넘어가. 삭제는 세 경우를 각각 다 그려볼 거야.

## 진행 순서
**이진 트리 개념(1~4) → BST 조건(5~7) → 검색 손추적(8~10) → 삽입 손추적(11~12) → 삭제 3경우 손추적(13~18) 🔥 → 코드 구현(19~24) → 성능과 균형(25~27)**

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
from __future__ import annotations
from typing import Any
import math, random, sys, time
sys.setrecursionlimit(300000)

# ===== 교재 실습 9-1 (+ 보충수업 9-2 내림차순) =====
class Node:
    """이진 검색 트리의 노드"""
    def __init__(self, key: Any, value: Any, left: Node = None, right: Node = None):
        self.key = key        # 키
        self.value = value    # 값
        self.left = left      # 왼쪽 포인터
        self.right = right    # 오른쪽 포인터


class BinarySearchTree:
    """이진 검색 트리"""
    def __init__(self):
        self.root = None

    def search(self, key: Any) -> Any:
        p = self.root
        while True:
            if p is None:  return None
            if key == p.key: return p.value
            elif key < p.key: p = p.left
            else: p = p.right

    def add(self, key: Any, value: Any) -> bool:
        def add_node(node: Node, key: Any, value: Any) -> bool:
            if key == node.key:
                return False
            elif key < node.key:
                if node.left is None: node.left = Node(key, value, None, None)
                else: return add_node(node.left, key, value)
            else:
                if node.right is None: node.right = Node(key, value, None, None)
                else: return add_node(node.right, key, value)
            return True
        if self.root is None:
            self.root = Node(key, value, None, None)
            return True
        else:
            return add_node(self.root, key, value)

    def remove(self, key: Any) -> bool:
        p = self.root          # 스캔 중인 노드
        parent = None          # 스캔 중인 노드의 부모 노드
        is_left_child = True   # p는 parent의 왼쪽 자식인지

        while True:                                  # ── 삭제할 키를 검색
            if p is None: return False
            if key == p.key: break
            else:
                parent = p
                if key < p.key: is_left_child = True;  p = p.left
                else:           is_left_child = False; p = p.right

        if p.left is None:                           # ── A/B: 왼쪽 자식이 없음
            if p is self.root:      self.root = p.right
            elif is_left_child:     parent.left = p.right
            else:                   parent.right = p.right
        elif p.right is None:                        # ── A/B: 오른쪽 자식이 없음
            if p is self.root:      self.root = p.left
            elif is_left_child:     parent.left = p.left
            else:                   parent.right = p.left
        else:                                        # ── C: 자식이 2개
            parent = p
            left = p.left
            is_left_child = True
            while left.right is not None:            # 왼쪽 서브트리의 최댓값 찾기
                parent = left
                left = left.right
                is_left_child = False
            p.key = left.key                         # left를 p로 이동
            p.value = left.value
            if is_left_child: parent.left = left.left
            else:             parent.right = left.left
        return True

    def dump(self, reverse=False) -> None:
        def go(node):
            if node is not None:
                go(node.left); print(f'{node.key} {node.value}'); go(node.right)
        def go_rev(node):
            if node is not None:
                go_rev(node.right); print(f'{node.key} {node.value}'); go_rev(node.left)
        go_rev(self.root) if reverse else go(self.root)

    def min_key(self) -> Any:
        if self.root is None: return None
        p = self.root
        while p.left is not None: p = p.left
        return p.key

    def max_key(self) -> Any:
        if self.root is None: return None
        p = self.root
        while p.right is not None: p = p.right
        return p.key


# ===== 실험용 도구 =====
def build(seq, val=lambda k: k*10):
    """키 리스트로 BST를 만든다 (값은 기본 키×10)"""
    t = BinarySearchTree()
    for k in seq: t.add(k, val(k))
    return t

def show(t, title=""):
    """트리를 옆으로 눕혀 출력 (오른쪽이 위) — 90도 돌려 보면 정상"""
    if title: print(title)
    def go(n, d=0):
        if n is None: return
        go(n.right, d+1)
        print('      '*d + str(n.key))
        go(n.left, d+1)
    go(t.root)
    print()

def inorder(t):
    r=[]
    def go(n):
        if n: go(n.left); r.append(n.key); go(n.right)
    go(t.root); return r

def kv(t):
    r=[]
    def go(n):
        if n: go(n.left); r.append(f"{n.key}:{n.value}"); go(n.right)
    go(t.root); return r

def height(t):
    def go(n): return -1 if n is None else 1+max(go(n.left), go(n.right))
    return go(t.root)

def trace_search(t, key):
    """검색 경로를 반환"""
    p, path = t.root, []
    while p is not None:
        path.append(p.key)
        if key == p.key: return path, True
        p = p.left if key < p.key else p.right
    return path, False

print("준비 완료 ✅\n")
show(build([11,5,15,4,7,13,18,1,6,9,12,14]), "예시: 교재 [그림 9-8]의 이진 검색 트리")

---

# 🔁 [Remind] 워밍업 — 32일차 & 12일차 소환

### R-1. 🟢 [설명] 어제의 중위 순회

32일차 25번에서 이 트리의 중위 순회를 해봤지:
```
            50
          ／    ＼
        30        70
       ／ ＼      ／ ＼
     20    40   60    80
```
- 중위 순회 결과는? ①________________
- 뭐가 특별했지? ②________________
- 오늘 이 성질이 **어느 함수**로 구현될까? ③____

### R-2. 🟡 [설명] 12일차 이진 검색과의 관계

12일차 이진 검색은 **정렬된 배열**에서 이렇게 했어:
```python
while pl <= pr:
    pc = (pl + pr) // 2
    if a[pc] == key: return pc
    elif a[pc] < key: pl = pc + 1
    else: pr = pc - 1
```
- 한 번 비교할 때마다 후보가 ①____ 로 줄었지. 시간 복잡도는? ②____
- 오늘 BST 검색도 똑같이 "비교하고 한쪽만 본다"인데, **배열의 `pc = (pl+pr)//2`** 에 해당하는 게 트리에서는 뭐야? ③____
- 🔥 그럼 **BST가 이진 검색보다 나은 점**은 뭘까? (힌트: 배열에 새 값을 삽입하려면?) ④________________

*(답을 적은 뒤 아래로)*

---

---

# 🌲 PART 1 — 이진 트리와 완전 이진 트리 (1~4번)

### 1. 🟢 [설명] 이진 트리

교재 382p:
> **"노드가 왼쪽 자식(left child)과 오른쪽 자식(right child)만을 갖는 트리를 이진 트리(binary tree)라고 합니다. 이때 두 자식 가운데 하나 또는 둘 다 존재하지 않는 노드가 있어도 상관없습니다."**

- 이진 트리의 조건을 한 문장으로: ①________________
- 32일차 7번에서 **차수**를 배웠지. 이진 트리를 차수로 정의하면? ②________________
- 🔥 **"하나 또는 둘 다 없어도 상관없다"** 는 게 왜 중요할까? 자식이 1개인 노드도 이진 트리에 속해? ③____

교재 382p: **"이진 트리의 특징은 왼쪽 자식과 오른쪽 자식을 구분하는 점입니다."**
- 32일차 11번의 **순서 트리 / 무순서 트리** 중 이진 트리는 어느 쪽이야? ④____
- 왼쪽과 오른쪽을 구분하지 않으면 오늘 배울 BST가 성립할 수 있을까? ⑤____ 왜?

---

### 2. 🟢 [설명] 왼쪽 서브트리 / 오른쪽 서브트리

교재 382p: **"왼쪽 자식을 루트로 하는 서브트리를 왼쪽 서브트리(left subtree)라 하고, 오른쪽 자식을 루트로 하는 서브트리를 오른쪽 서브트리(right subtree)라고 합니다."**

부록에서 출력한 [그림 9-8] 트리를 봐:
```
            11
          ／    ＼
        5         15
       ／ ＼      ／  ＼
     4     7    13     18
    ／    ／ ＼  ／ ＼
   1     6    9 12   14
```

- `5` 의 왼쪽 서브트리 노드들: ①________
- `5` 의 오른쪽 서브트리 노드들: ②________
- `15` 의 왼쪽 서브트리: ③________
- `11` 의 왼쪽 서브트리 노드 개수: ④____ / 오른쪽: ⑤____
- 🔥 리프의 왼쪽/오른쪽 서브트리는? ⑥________________ (32일차 10번의 **빈 트리**!)

---

### 3. 🟡 [설명] 완전 이진 트리

교재 382p:
> **"루트부터 아래쪽 레벨로 노드가 가득 차 있고, 같은 레벨 안에서 왼쪽부터 오른쪽으로 노드가 채워져 있는 이진 트리를 완전 이진 트리(complete binary tree)라고 합니다."**
> - 마지막 레벨을 제외하고 모든 레벨에 노드가 가득 차 있습니다.
> - 마지막 레벨에 한해서 왼쪽부터 오른쪽으로 노드를 채우되 반드시 끝까지 채우지 않아도 됩니다.

**다음 중 완전 이진 트리를 전부 골라봐:**

**(가)**
```
      A
     / \
    B   C
   / \
  D   E
```
**(나)**
```
      A
     / \
    B   C
   /     \
  D       E
```
**(다)**
```
      A
     / \
    B   C
   / \  /
  D  E F
```
**(라)**
```
      A
     /
    B
```
**(마)**
```
      A
       \
        B
```

| | 완전 이진 트리? | 이유 |
|---|---|---|
| (가) | ① | ② |
| (나) | ③ | ④ |
| (다) | ⑤ | ⑥ |
| (라) | ⑦ | ⑧ |
| (마) | ⑨ | ⑩ |

- 🔥 **(나)와 (마)가 왜 안 될까?** 공통점을 찾아봐: ⑪________________

---

### 4. 🟡 [설명] 완전 이진 트리와 배열 — 24일차 재회

교재 383p:
> **"[그림 9-7]과 같이 너비 우선 검색에서 스캔하는 순서대로 각 노드에 0, 1, 2, …의 값을 주면 배열에 저장하는 인덱스와 일대일로 정확히 대응시킬 수 있습니다."**
> **"이 방법은 06장에서 학습한 '힙 정렬'에서 사용했습니다."**

- 24일차 4번의 인덱스 공식 세 개를 다시 적어봐:
  - 부모: ①____ / 왼쪽 자식: ②____ / 오른쪽 자식: ③____
- 32일차 13번의 **너비 우선 검색** 순서와 배열 인덱스가 왜 일치하지? ④________________
- 🔥 **완전 이진 트리가 아니면** 이 대응이 왜 깨질까? ⑤________________
  💡 힌트: 29일차 5번에서 봤던 "중간이 빈 트리를 배열에 담으면?"
- **높이가 k인 완전 이진 트리의 최대 노드 수는?** ⑥____
- 그럼 **노드 n개를 담는 완전 이진 트리의 높이는?** ⑦____
- 🔥 이 두 식이 서로 **역함수** 관계인 걸 확인해봐. `n = 2^(k+1) - 1` 을 `k` 에 대해 풀면? ⑧________________

*(답을 적은 뒤 실행)*

In [ ]:
print("높이 k | 최대 노드 수 n = 2^(k+1) - 1 | k = log2(n+1) - 1")
print("-" * 58)
for k in (0, 1, 2, 3, 5, 10, 20, 30):
    n = 2**(k+1) - 1
    print(f"  {k:2d}   | {n:>15,d}          | {math.log2(n+1)-1:6.1f}")

print("\n🔥 높이 30이면 21억 개를 담는다")
print("   = 21억 개짜리 트리도 루트에서 리프까지 30번이면 도달")
print("   → 12일차 이진 검색이 21억 개를 31번에 찾던 것과 같은 원리\n")

print("[k가 1 늘 때마다 n은 2배]")
prev = None
for k in range(0, 6):
    n = 2**(k+1) - 1
    r = f"  (직전의 {n/prev:.1f}배)" if prev else ""
    print(f"  k={k}: n={n:3d}{r}")
    prev = n
print("\n→ n이 지수적으로 커지니, 거꾸로 k는 로그적으로 커진다 = O(log n)")

---

# 🔑 PART 2 — 이진 검색 트리의 조건 (5~7번)

### 5. 🟢 [설명] 두 줄짜리 규칙

교재 384p:
> **"이진 검색 트리(binary search tree)는 모든 노드가 다음 조건을 만족해야 합니다."**
> - **왼쪽 서브트리 노드의 키값은 자신의 노드 키값보다 작아야 합니다.**
> - **오른쪽 서브트리 노드의 키값은 자신의 노드 키값보다 커야 합니다.**

🔥 **"자식"이 아니라 "서브트리"** 라는 게 결정적이야.

[그림 9-8] 트리에서 노드 `5` 를 봐:
```
            11
          ／    ＼
        5         15
       ／ ＼      ／  ＼
     4     7    13     18
    ／    ／ ＼  ／ ＼
   1     6    9 12   14
```
- `5` 의 왼쪽 서브트리는 `{4, 1}` 이고 **둘 다 5보다 작아** ✅
- `5` 의 오른쪽 서브트리는 `{7, 6, 9}` 이고 **셋 다 5보다 커** ✅

**질문:**
- 교재 384p: **"따라서 키값이 같은 노드는 복수로 존재하지 않습니다."** 왜 그럴까? ①________________
- `11` 의 왼쪽 서브트리 6개 노드가 전부 11보다 작은지 확인해봐: ②________
- 🔥 만약 조건이 **"자식만"** 이라면 어떤 문제가 생길까? 아래 6번에서 반례를 찾아봐.

---

### 6. 🔴 [판별] 🔥 BST인가 아닌가

**"자식만 보면 안 된다"** 는 걸 보여주는 함정 문제야.

**(가)**
```
      8
     / \
    3   10
   / \    \
  1   6    14
     / \   /
    4   7 13
```

**(나)** 🔥 잘 봐
```
      8
     / \
    3   10
   / \
  1   6
     / \
    4   9      ← 9!
```

**(다)**
```
      5
     / \
    3   7
   /     \
  1       9
```

**(라)** 🔥
```
      5
     / \
    3   7
     \   \
      6   9    ← 6!
```

| | BST? | 위반한 노드 쌍 |
|---|---|---|
| (가) | ① | ② |
| (나) | ③ | ④ |
| (다) | ⑤ | ⑥ |
| (라) | ⑦ | ⑧ |

- 🔥 **(나)와 (라)의 공통점**: 부모-자식만 보면 규칙에 맞는데 **조상까지 보면 틀려.** 각각 어떤 조상과 충돌해? ⑨________________
- 그래서 BST를 검증하는 올바른 방법은 **"각 노드가 가질 수 있는 값의 범위"** 를 위에서 아래로 전달하는 거야. 예를 들어 (나)의 `9` 는 어떤 범위 안에 있어야 했지? ⑩________

*(답을 적은 뒤 실행)*

In [ ]:
def is_bst_wrong(node):
    """❌ 자식만 검사하는 잘못된 방법"""
    if node is None: return True
    if node.left and node.left.key >= node.key: return False
    if node.right and node.right.key <= node.key: return False
    return is_bst_wrong(node.left) and is_bst_wrong(node.right)

def is_bst_right(node, lo=None, hi=None):
    """✅ 범위를 물려주는 올바른 방법"""
    if node is None: return True
    if lo is not None and node.key <= lo: return False
    if hi is not None and node.key >= hi: return False
    return (is_bst_right(node.left, lo, node.key) and
            is_bst_right(node.right, node.key, hi))

def manual(pairs, root):
    """(부모키, 자식키, 'L'/'R') 목록으로 트리를 직접 만든다"""
    nodes = {}
    def get(k):
        if k not in nodes: nodes[k] = Node(k, k*10, None, None)
        return nodes[k]
    r = get(root)
    for p, c, side in pairs:
        if side == 'L': get(p).left = get(c)
        else:           get(p).right = get(c)
    t = BinarySearchTree(); t.root = r
    return t

cases = {
 "(가)": manual([(8,3,'L'),(8,10,'R'),(3,1,'L'),(3,6,'R'),(10,14,'R'),
                 (6,4,'L'),(6,7,'R'),(14,13,'L')], 8),
 "(나)": manual([(8,3,'L'),(8,10,'R'),(3,1,'L'),(3,6,'R'),(6,4,'L'),(6,9,'R')], 8),
 "(다)": manual([(5,3,'L'),(5,7,'R'),(3,1,'L'),(7,9,'R')], 5),
 "(라)": manual([(5,3,'L'),(5,7,'R'),(3,6,'R'),(7,9,'R')], 5),
}
print(f"{'':6s} {'자식만 검사':>12s} {'범위 검사(정답)':>16s}   중위 순회")
print("-"*62)
for nm, t in cases.items():
    w = is_bst_wrong(t.root); r = is_bst_right(t.root)
    mark = "  ← 자식만 보면 속는다! 🔥" if w != r else ""
    print(f"{nm:6s} {str(w):>12s} {str(r):>16s}   {inorder(t)}{mark}")

print("\n💡 검증 팁: BST라면 중위 순회가 반드시 '오름차순'이어야 한다")
print("   (나)와 (라)의 중위 순회를 보면 순서가 깨진 게 보이지?")

### 7. 🟡 [설명] BST의 네 가지 특징

교재 384p가 정리한 특징이야. 각각 **왜** 그런지 채워봐.

| 특징 | 이유 |
|---|---|
| ① **구조가 단순합니다** | 노드에 필요한 필드가 `key`, `value`, `left`, `right` 넷뿐 |
| ② **중위 순회의 깊이 우선 검색을 통하여 노드값을 오름차순으로 얻을 수 있습니다** | ⓐ________________ |
| ③ **이진 검색과 비슷한 방식으로 아주 빠르게 검색할 수 있습니다** | ⓑ________________ |
| ④ **노드를 삽입하기 쉽습니다** | ⓒ________________ |

**교재 384p의 중위 순회 결과를 직접 확인해봐:**
```
1 → 4 → 5 → 6 → 7 → 9 → 11 → 12 → 13 → 14 → 15 → 18
```
- 부록 셀에서 출력한 트리와 일치해? ⓓ____
- 🔥 ④번(삽입이 쉽다)을 **정렬된 배열**과 비교해봐. 배열 한가운데에 값을 넣으려면? ⓔ________________ (29일차 1번!)

---

---

# 🔍 PART 3 — 검색을 손으로 (8~10번)

## 📌 검색·삽입용 기준 트리 (트리 ①)

교재 [그림 9-11], [그림 9-12]와 같은 트리야.

```
          5
        ／   ＼
      2        7
    ／  ＼
   1      4
        ／
       3
```

### 8. 🟢 [손] 검색 성공 — 교재 그림 9-11

교재 386p가 `3` 을 검색하는 과정을 4단계로 보여줘:

> 1. 처음에 주목하는 루트의 키는 5입니다. 3은 5보다 작으므로 **왼쪽** 자식 노드를 따라갑니다.
> 2. 다음에 주목하는 노드의 키는 2입니다. 3은 2보다 크므로 **오른쪽** 자식 노드를 따라갑니다.
> 3. 다음에 주목하는 노드의 키는 4입니다. 3은 4보다 작으므로 **왼쪽** 자식 노드를 따라갑니다.
> 4. 키가 3인 노드에 도달했습니다. 검색에 **성공**합니다.

**표를 채워봐:**

| 단계 | 주목 노드 p | `key(3)` vs `p.key` | 다음 동작 |
|---|---|---|---|
| 1 | 5 | 3 < 5 | ① |
| 2 | ② | ③ | ④ |
| 3 | ⑤ | ⑥ | ⑦ |
| 4 | ⑧ | ⑨ | ⑩ |

- 총 **몇 번 비교**했어? ⑪____
- 트리의 높이는 ⑫____ 인데, 최대 비교 횟수는 **높이 + 1** 이지? 왜 +1일까? ⑬________________

---

### 9. 🟡 [손] 검색 실패 — 교재 그림 9-12

교재 387p가 `8` 을 검색하는 과정:

> 1. 처음에 주목하는 루트의 키는 5입니다. 8은 5보다 크므로 **오른쪽** 자식 노드를 따라갑니다.
> 2. 다음에 주목하는 노드의 키는 7입니다. 주목 노드는 리프이고 오른쪽 자식 노드는 존재하지 않습니다. 더 이상 스캔을 할 수 없으므로 검색에 **실패**합니다.

**표를 채워봐:**

| 단계 | 주목 노드 p | 비교 | 다음 |
|---|---|---|---|
| 1 | 5 | 8 > 5 | ① |
| 2 | ② | ③ | ④ **p = None** |
| 3 | ⑤ | - | ⑥ |

- 실패를 감지하는 코드 줄은? ⑦________________
- 🔥 **같은 트리에서 다음 키들을 검색하면 경로가 어떻게 될까?** 직접 써봐:

| 검색 키 | 경로 | 결과 |
|---|---|---|
| 1 | ⑧ | ⑨ |
| 4 | ⑩ | ⑪ |
| 6 | ⑫ | ⑬ |
| 0 | ⑭ | ⑮ |

*(답을 적은 뒤 실행)*

In [ ]:
t1 = build([5,2,7,1,4,3])
show(t1, "트리 ① (오른쪽이 위 — 90도 돌려서 봐)")
print("중위 순회:", inorder(t1), "← 오름차순 ✅\n")

for key in (3, 8, 1, 4, 6, 0):
    path, ok = trace_search(t1, key)
    arrow = ' → '.join(map(str, path))
    print(f"  search({key:2d}): {arrow:20s} → {'성공 ✅' if ok else '실패 ❌'}  ({len(path)}번 비교)")

print(f"\n트리 높이 = {height(t1)}, 최대 비교 횟수 = {height(t1)+1}")

### 10. 🟡 [설명] 검색 알고리즘 정리

교재 387p가 정리한 알고리즘이야. 빈칸을 채워봐.

```
1. 루트에 주목합니다. 여기서 주목하는 노드를 p라고 하겠습니다.
2. p가 ①____ 이면 검색을 실패하고 종료합니다.
3. 검색하는 key와 주목 노드 p의 키를 비교합니다.
   · key = p: ②________________
   · key < p: ③________________
   · key > p: ④________________
4. ⑤____ 번 과정으로 되돌아갑니다.
```

- 🔥 이 알고리즘이 **12일차 이진 검색**과 다른 점 하나만 꼽으면? ⑥________________
  💡 힌트: 이진 검색은 `pl`, `pr` 로 **범위**를 좁혔는데, BST는?
- **한 번 비교할 때마다 후보가 얼마나 줄어?** ⑦________________
  ⚠️ 단, 이건 트리가 **균형 잡혀 있을 때**만 성립해 (25~27번에서!)

---

---

# ➕ PART 4 — 삽입을 손으로 (11~12번)

### 11. 🟢 [손] 삽입 과정 — 교재 그림 9-13

교재 388p:
> **"노드를 삽입할 때 주의할 점은 노드를 삽입한 뒤에 트리의 형태가 이진 검색 트리의 조건을 유지해야 한다는 것입니다. 따라서 노드를 삽입할 때에는 검색할 때와 마찬가지로 먼저 삽입할 위치를 찾아낸 뒤에 수행해야 합니다."**

**ⓐ 노드 4개(2, 4, 6, 7)로 구성된 트리에 `1` 을 삽입**
```
        6
      ／  ＼
    2       7
      ＼
        4
```
> 1. 삽입할 위치를 찾습니다. 추가할 값 1은 2보다 작고 **왼쪽 자식 노드가 존재하지 않으므로** 삽입할 위치로 2를 선택합니다.
> 2. 1을 2의 **왼쪽** 자식 노드로 삽입합니다.

- 경로를 써봐: `6` → ①____ → 왼쪽 자식 자리가 비어 있음 → 삽입
- 삽입 후 트리를 그려봐

**ⓑ 이어서 `5` 를 삽입**
> 1. 추가할 값 5는 4보다 크고 오른쪽 자식 노드가 존재하지 않으므로 삽입할 위치로 4를 선택합니다.
> 2. 5를 4의 **오른쪽** 자식 노드로 삽입합니다.

- 경로: `6` → ②____ → ③____ → 오른쪽 자리 비어 있음 → 삽입
- 최종 중위 순회: ④________________

🔥 **핵심**: 삽입은 **항상 리프 자리**에 일어나. 중간에 끼워 넣는 일이 없어.
- 왜 그럴까? ⑤________________
- 그럼 **29일차 1번의 "배열은 삽입할 때 뒤를 다 밀어야 한다"** 문제가 BST에는 있을까? ⑥____

---

### 12. 🟡 [설명] 삽입 알고리즘과 재귀

교재 389p 알고리즘:
```
1. 루트에 주목합니다. 여기서 주목하는 노드를 node라고 하겠습니다.
2. 삽입하는 key와 주목 노드 node의 키를 비교합니다.
   · key = node인 경우: ①________________
   · key < node인 경우:
       - 왼쪽 자식 노드가 없으면 ②________________
       - 왼쪽 자식 노드가 있으면 ③________________
   · key > node:
       - 오른쪽 자식 노드가 없으면 ④________________
       - 오른쪽 자식 노드가 있으면 ⑤________________
3. 2번 과정으로 되돌아갑니다.
```

- 🔥 `key == node.key` 일 때 **삽입 실패(False 반환)** 인 이유는? (5번의 "키값이 같은 노드는 복수로 존재하지 않습니다") ⑥________________
- `add` 는 **바깥 함수 + 내부 재귀 함수** 구조야. 바깥이 처리하는 특수 상황은? ⑦________________
- 🔥 **루트가 None일 때만 바깥에서 처리하는 이유**는? 내부 함수 `add_node(node, ...)` 는 `node` 가 절대 `None` 이 아니라고 가정하거든. 왜 그렇게 설계했을까? ⑧________________

*(답을 적은 뒤 실행)*

In [ ]:
t = build([6,2,7,4])
show(t, "ⓐ 삽입 전 (2, 4, 6, 7)")
print("경로 추적: 1을 삽입하려면")
p = t.root
while p:
    print(f"  {p.key} 와 비교 → 1 < {p.key} 이므로 왼쪽" if 1 < p.key else f"  {p.key} 와 비교 → 오른쪽")
    nxt = p.left if 1 < p.key else p.right
    if nxt is None:
        print(f"  → {p.key} 의 {'왼쪽' if 1 < p.key else '오른쪽'} 자리가 비어 있다! 여기 삽입\n")
        break
    p = nxt
t.add(1, 10)
show(t, "ⓐ 1 삽입 후")

t.add(5, 50)
show(t, "ⓑ 5 삽입 후")
print("중위 순회:", inorder(t))

print("\n[중복 키 삽입]")
print("  add(5, 999) →", t.add(5, 999), " ← False! (이미 존재)")
print("  값도 안 바뀌었나?", [x for x in kv(t) if x.startswith('5:')])

print("\n[삽입은 항상 리프 자리에]")
print("  삽입 위치를 찾는 과정 = 검색 과정과 동일")
print("  검색이 실패하는 지점(None을 만나는 자리)이 곧 삽입할 자리다 🔥")

---

# ➖ PART 5 — 삭제 3경우를 손으로 (13~18번)

> **오늘의 하이라이트.** 교재 390p: **"노드를 삭제하는 과정은 삽입하는 과정보다 복잡합니다."**
>
> 왜 복잡할까? **삭제한 자리를 누군가 메워야** 하는데, 그게 **자식 수에 따라 달라지기 때문**이야.
> 32일차 7번에서 배운 **차수** 개념이 여기서 정확히 쓰여.

## 📌 삭제용 기준 트리 (트리 ②)

교재 [그림 9-15], [그림 9-16]과 같은 트리야.

```
              6
           ／     ＼
         2           7
       ／   ＼          ＼
     1        4           8
   ／       ／  ＼           ＼
  0       3      5            9
```

**중위 순회**: `0 1 2 3 4 5 6 7 8 9`

각 노드의 **차수**를 먼저 적어봐:

| 노드 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| 차수 | | | | | | | | | | |

- 차수 0(리프)인 노드: ①________
- 차수 1인 노드: ②________
- 차수 2인 노드: ③________

---

### 13. 🟢 [손] 경우 A — 자식이 없는 노드 삭제

교재 390p [그림 9-15]:
> **"노드 3을 가리키는 부모 노드 4의 왼쪽 포인터가 노드 3을 가리키지 않도록 업데이트합니다. 즉, 왼쪽 포인터를 None으로 합니다. 그 결과 노드 3을 가리키는 노드가 없기 때문에 이진 검색 트리에서 삭제됩니다."**

교재가 정리한 규칙:
> - 삭제할 노드가 부모 노드의 **왼쪽** 자식이면, 부모의 **왼쪽 포인터**를 None으로 합니다.
> - 삭제할 노드가 부모 노드의 **오른쪽** 자식이면, 부모의 **오른쪽 포인터**를 None으로 합니다.

**ⓐ `3` 을 삭제**
- `3` 을 찾아가는 경로: ①________________
- `3` 의 부모는? ②____ `3` 은 부모의 **왼쪽/오른쪽** 자식? ③____
- 그럼 `4.____ = None` ④____
- 삭제 후 중위 순회: ⑤________________

**ⓑ `9` 를 삭제**
- 경로: ⑥________________
- `9` 의 부모는 ⑦____, `9` 는 부모의 ⑧____ 자식
- `8.____ = None` ⑨____
- 삭제 후 중위 순회: ⑩________________

🔥 **"삭제"의 정체**를 29일차 9번과 연결해봐: ⑪________________

---

### 14. 🟡 [손] 경우 B — 자식이 1개인 노드 삭제

교재 391p [그림 9-16]:
> **"원래 노드 7의 위치에 노드 8을 가져오면 삭제할 수 있습니다. 왜냐하면 '자식 노드 8을 루트로 하는 서브트리의 모든 키는 부모 노드 6보다 커야 한다'는 관계가 성립하기 때문입니다."**

교재 규칙:
> - 삭제할 노드가 부모 노드의 **왼쪽** 자식인 경우: 부모의 왼쪽 포인터가 **삭제할 노드의 자식**을 가리키도록 업데이트합니다.
> - 삭제할 노드가 부모 노드의 **오른쪽** 자식인 경우: 부모의 오른쪽 포인터가 **삭제할 노드의 자식**을 가리키도록 업데이트합니다.

**ⓐ `7` 을 삭제** (`7` 은 오른쪽 자식 `8` 만 있음)
- `7` 의 부모는 ①____, `7` 은 부모의 ②____ 자식
- `7` 의 자식은 ③____
- 따라서 `6.____ = ____` ④________
- 삭제 후 중위 순회: ⑤________________
- 🔥 **`8` 의 서브트리 전체(8, 9)가 `6` 보다 크다**는 게 왜 중요해? ⑥________________

**ⓑ `1` 을 삭제** (`1` 은 왼쪽 자식 `0` 만 있음)
- `1` 의 부모는 ⑦____, `1` 은 부모의 ⑧____ 자식
- `2.____ = ____` ⑨________
- 삭제 후 중위 순회: ⑩________________

🔥 **경우 A와 B를 같은 코드로 처리할 수 있어.** 왜일까?
```python
if p.left is None:            # 왼쪽 자식이 없음
    ... = p.right             # 오른쪽 자식(또는 None)을 물려줌
elif p.right is None:         # 오른쪽 자식이 없음
    ... = p.left              # 왼쪽 자식(또는 None)을 물려줌
```
- 자식이 **0개**인 노드는 `p.left` 도 `p.right` 도 `None` 이야. 그럼 위 코드에서 어느 분기를 타? ⑪____
- 그때 물려주는 값은? ⑫____ → 결과적으로 **경우 A와 정확히 같은 동작** ✅
- 교재 394p 설명: **"A와 B를 같은 순서로 수행하는 것은 삭제 노드에 왼쪽 자식이 없으면 왼쪽 포인터가 None이 되고, 오른쪽 자식이 없으면 오른쪽 포인터가 None이 된다는 것을 이용하기 때문입니다."**

*(답을 적은 뒤 실행)*

In [ ]:
seq = [6,2,7,1,4,8,3,5,9,0]
t = build(seq)
show(t, "트리 ② (삭제 전)")
print("중위:", inorder(t))
print("각 노드의 차수:")
def deg(n): return (1 if n.left else 0) + (1 if n.right else 0)
def walk(n, out):
    if n: walk(n.left,out); out.append((n.key,deg(n))); walk(n.right,out)
o=[]; walk(t.root,o)
print("  " + "  ".join(f"{k}:{d}" for k,d in o))
print("  차수0(리프):", [k for k,d in o if d==0])
print("  차수1      :", [k for k,d in o if d==1])
print("  차수2      :", [k for k,d in o if d==2])

print("\n" + "="*56)
for key, label in [(3,"[A-ⓐ] 3 삭제 (자식 0개)"), (9,"[A-ⓑ] 9 삭제 (자식 0개)"),
                   (7,"[B-ⓐ] 7 삭제 (자식 1개: 8)"), (1,"[B-ⓑ] 1 삭제 (자식 1개: 0)")]:
    t = build(seq)
    path,_ = trace_search(t, key)
    t.remove(key)
    print(f"{label}")
    print(f"  경로: {' → '.join(map(str,path))}")
    print(f"  중위: {inorder(t)}\n")

print("[B-ⓐ] 7 삭제 후 트리 모양")
t = build(seq); t.remove(7)
show(t)

### 15. 🔴 [손] 경우 C — 자식이 2개인 노드 삭제 🔥🔥

교재 392p:
> **"자식 노드가 2개인 노드를 삭제하는 과정은 앞의 A, B 경우보다 복잡합니다."**

## 📌 경우 C용 기준 트리 (트리 ③) — 교재 [그림 9-17]

**네가 직접 테스트했던 그 트리야!**

```
                9
             ／     ＼
           5           10
         ／   ＼            ＼
       2        7             11
     ／  ＼   ／   ＼             ＼
    1     4  6      8              12
        ／
       3
```

교재가 정리한 순서:
> 1. **삭제할 노드의 왼쪽 서브트리에서 키값이 가장 큰 노드를 검색합니다.**
> 2. **검색한 노드를 삭제 위치로 옮깁니다.** 즉, 검색한 노드의 데이터를 삭제할 노드 위치에 복사합니다.
> 3. **옮긴 노드를 삭제합니다.** 이때 자식 노드의 개수에 따라 다음을 수행합니다.
>    - 옮긴 노드에 자식이 없으면 **A**에 따라 삭제합니다.
>    - 옮긴 노드에 자식이 1개만 있으면 **B**에 따라 삭제합니다.

**`5` 를 삭제해보자.**

**1단계 — 왼쪽 서브트리에서 최댓값 찾기**
- `5` 의 왼쪽 서브트리의 루트는? ①____
- 그 서브트리의 노드들: ②________
- 그중 **가장 큰 값**은? ③____
- 🔥 **최댓값은 어떻게 찾지?** 32일차 25번에서 배웠어: ④________________

**2단계 — 복사**
- `5` 자리에 ⑤____ 의 **키와 값**을 복사
- 🔥 **왜 노드 자체를 옮기지 않고 데이터만 복사할까?** ⑥________________

**3단계 — 옮긴 노드 삭제**
- 원래 `4` 가 있던 자리를 지워야 해. `4` 의 자식은? ⑦____
- 자식이 1개니까 **경우 ⑧____** 로 처리
- `4` 의 부모는 ⑨____, `4` 는 부모의 ⑩____ 자식
- 따라서 `2.____ = ____` ⑪________

**최종 트리를 그려봐:**
```
                9
             ／     ＼
          ⑫           10
         ／  ＼            ＼
       2       7             11
     ／ ＼    ／  ＼             ＼
    1   ⑬   6     8              12
```

- 삭제 후 중위 순회: ⑭________________
- 🔥 **`4` 의 value 는 얼마여야 해?** ⑮____ (테스트에서 `4:40` 이 나와야 정상!)

*(답을 적은 뒤 실행)*

In [ ]:
seq3 = [9,5,10,2,7,11,1,4,6,8,12,3]
t = build(seq3)
show(t, "트리 ③ (교재 그림 9-17) — 삭제 전")
print("중위:", inorder(t), "\n")

print("[1단계] 5의 왼쪽 서브트리에서 최댓값 찾기")
p = t.root.left                      # 노드 5
print(f"  삭제 대상 p = {p.key}")
left = p.left                        # 왼쪽 서브트리 루트
print(f"  왼쪽 서브트리 루트 = {left.key}")
parent = p; is_left = True
path = [left.key]
while left.right is not None:
    parent = left; left = left.right; is_left = False
    path.append(left.key)
print(f"  오른쪽으로만 계속: {' → '.join(map(str,path))}")
print(f"  → 최댓값 = {left.key}, 그 부모 = {parent.key}, 왼쪽 자식? {is_left}")
print(f"  → {left.key} 의 자식: 왼쪽={left.left.key if left.left else None}, "
      f"오른쪽={left.right.key if left.right else None}\n")

t.remove(5)
show(t, "[2~3단계] 5 삭제 후")
print("중위:", inorder(t))
print("key:value 전체:")
print("  " + " | ".join(kv(t)))
print("\n🔥 '4:40' 인지 확인! '4:50' 이면 p.value = left.value 를 빠뜨린 버그")

### 16. 🔴 [설명] 왜 하필 "왼쪽 서브트리의 최댓값"일까

교재는 **"왼쪽 서브트리에서 가장 큰 노드"** 를 쓰라고 해. **왜 그게 정답일까?**

삭제할 노드를 `p` 라고 하자. `p` 자리에 올 수 있는 값의 조건은:
- `p` 의 **왼쪽 서브트리 전체보다 크거나 같아야** 한다 (안 그러면 BST 조건 깨짐)
- `p` 의 **오른쪽 서브트리 전체보다 작거나 같아야** 한다

**질문에 답해봐:**
- 이 조건을 만족하는 값이 트리 안에 **몇 개** 있을까? ①____
  💡 힌트: 중위 순회에서 `p` 의 **바로 앞**과 **바로 뒤**를 생각해봐
- **왼쪽 서브트리의 최댓값** = 중위 순회에서 `p` 의 ②________ (선행자, predecessor)
- **오른쪽 서브트리의 최솟값** = 중위 순회에서 `p` 의 ③________ (후행자, successor)
- 🔥 그럼 **둘 중 아무거나 써도 될까?** ④____ 교재는 왼쪽 최댓값을 썼지만 오른쪽 최솟값을 써도 BST는 유지돼.

**트리 ③에서 확인해봐** (중위 순회: `1 2 3 4 5 6 7 8 9 10 11 12`)
- `5` 의 바로 앞(선행자) = ⑤____ ← 왼쪽 서브트리의 최댓값
- `5` 의 바로 뒤(후행자) = ⑥____ ← 오른쪽 서브트리의 최솟값
- 둘 다 `5` 자리에 와도 BST가 유지될까? 직접 그려서 확인해봐

**그리고 결정적으로:**
- 🔥 **"왼쪽 서브트리의 최댓값"은 오른쪽 자식이 없다는 게 보장돼.** 왜? ⑦________________
- 그래서 그 노드를 삭제할 때는 **경우 C가 절대 안 나와** — A 또는 B만 나오지. **재귀가 무한히 깊어지지 않는 이유**야.

*(답을 적은 뒤 실행)*

In [ ]:
seq3 = [9,5,10,2,7,11,1,4,6,8,12,3]
t = build(seq3)
ino = inorder(t)
print("트리 ③ 중위 순회:", ino)
i = ino.index(5)
print(f"\n  5의 선행자(바로 앞) = {ino[i-1]}  ← 왼쪽 서브트리의 최댓값")
print(f"  5의 후행자(바로 뒤) = {ino[i+1]}  ← 오른쪽 서브트리의 최솟값")

print("\n[왼쪽 최댓값 vs 오른쪽 최솟값 — 둘 다 되는가?]")
class BST2(BinarySearchTree):
    """오른쪽 서브트리의 최솟값을 쓰는 변형"""
    def remove(self, key):
        p=self.root; parent=None; is_left=True
        while True:
            if p is None: return False
            if key==p.key: break
            parent=p
            if key<p.key: is_left=True;  p=p.left
            else:         is_left=False; p=p.right
        if p.left is None:
            if p is self.root: self.root=p.right
            elif is_left: parent.left=p.right
            else: parent.right=p.right
        elif p.right is None:
            if p is self.root: self.root=p.left
            elif is_left: parent.left=p.left
            else: parent.right=p.left
        else:
            parent=p; right=p.right; is_left=False
            while right.left is not None:      # 오른쪽 서브트리의 최솟값
                parent=right; right=right.left; is_left=True
            p.key=right.key; p.value=right.value
            if is_left: parent.left=right.right
            else:       parent.right=right.right
        return True

a = build(seq3); a.remove(5)
b = BST2()
for k in seq3: b.add(k, k*10)
b.remove(5)
print("  교재 방식(왼쪽 최댓값 4가 올라옴):", inorder(a))
print("  변형 방식(오른쪽 최솟값 6이 올라옴):", inorder(b))
print("  → 중위 순회 결과가 같다! 둘 다 유효한 BST ✅\n")
show(a, "  교재 방식 트리")
show(b, "  변형 방식 트리")
print("🔥 모양은 다르지만 둘 다 BST 조건을 만족한다")

### 17. 🟡 [손] 루트 삭제 — 특수 케이스

15번에서 `5` 를 지운 트리에서 이번엔 **루트 `9`** 를 지워보자.

```
                9              ← 이걸 삭제
             ／     ＼
           4           10
         ／   ＼            ＼
       2        7             11
     ／  ＼   ／   ＼             ＼
    1     3  6      8              12
```

- `9` 의 차수는? ①____ → 경우 ②____
- `9` 의 왼쪽 서브트리 루트는 ③____, 그 서브트리의 최댓값은? ④____
  - 경로: `4` → 오른쪽 `7` → 오른쪽 `8` → 오른쪽 없음 → 최댓값 **8**
- `8` 의 자식은? ⑤____ → 삭제할 때 경우 ⑥____
- `8` 의 부모는 ⑦____, `8` 은 부모의 ⑧____ 자식
- 삭제 후 **새 루트**는? ⑨____
- 최종 중위 순회: ⑩________________

🔥 **루트를 삭제할 때 코드가 특별히 처리하는 게 있어.** 코드에서 찾아봐:
```python
if p is self.root:
    self.root = p.right      # 또는 p.left
```
- 이 분기가 **왜 필요할까?** 루트는 ⑪________________
- 29일차 11번의 `remove(p)` 에서 `if p is self.head:` 분기가 있었던 것과 같은 이유지?
- ⚠️ 그런데 **경우 C에서 루트를 삭제할 때는** 이 분기를 안 타. 왜일까? ⑫________________
  💡 힌트: 경우 C는 **노드를 지우는 게 아니라 데이터만 덮어쓰거든.**

*(답을 적은 뒤 실행)*

In [ ]:
seq3 = [9,5,10,2,7,11,1,4,6,8,12,3]
t = build(seq3)
t.remove(5)
print("5 삭제 후 (여기서 시작):", inorder(t))
show(t)

print("[9(루트) 삭제 추적]")
p = t.root
print(f"  삭제 대상 = 루트 {p.key}, 차수 = {(1 if p.left else 0)+(1 if p.right else 0)}")
parent=p; left=p.left; is_left=True; path=[left.key]
while left.right is not None:
    parent=left; left=left.right; is_left=False; path.append(left.key)
print(f"  왼쪽 서브트리 최댓값 탐색: {' → '.join(map(str,path))}")
print(f"  → 최댓값 {left.key}, 부모 {parent.key}, 자식: 왼쪽={left.left.key if left.left else None}")

t.remove(9)
show(t, "9 삭제 후")
print("중위:", inorder(t))
print(f"새 루트 = {t.root.key}  ← 노드 자체가 바뀐 게 아니라 데이터가 덮어써졌다 🔥")
print("  (루트 객체는 그대로, key/value만 8/80으로 교체됨)")

### 18. 🟡 [설명] 삭제 알고리즘 전체 정리

교재 393~394p 코드를 세 단계로 나눠 정리해봐.

**1단계 (070~083행): 삭제할 키를 검색**
```python
while True:
    if p is None: return False
    if key == p.key: break
    else:
        parent = p
        if key < p.key: is_left_child = True;  p = p.left
        else:           is_left_child = False; p = p.right
```
- 이 루프가 끝났을 때 `p` 는 ①____, `parent` 는 ②____
- 🔥 `is_left_child` 는 왜 필요해? ③________________
- 8~10번의 **검색** 코드와 비교하면 뭐가 추가됐어? ④________________

**2단계 (086~098행): 경우 A와 B**
```python
if p.left is None:            # ← ⑤ 어떤 경우들을 포함?
    ...
elif p.right is None:         # ← ⑥
    ...
```
- 각 분기 안에서 **세 갈래**로 또 나뉘어. 무엇에 따라? ⑦________________

**3단계 (100~113행): 경우 C**
```python
parent = p
left = p.left
is_left_child = True
while left.right is not None:     # ⑧ 무엇을 찾는 루프?
    parent = left
    left = left.right
    is_left_child = False
p.key = left.key                  # ⑨
p.value = left.value              # ⑩ ← 빠뜨리면?
if is_left_child: parent.left = left.left
else:             parent.right = left.left
```
- ⑧ 루프가 끝나면 `left` 는 ⑪____, `parent` 는 ⑫____
- 🔥 `is_left_child = True` 로 **초기화**하는 이유는? ⑬________________
  💡 힌트: `while` 이 **한 번도 안 도는 경우**가 있어. 언제?
- 마지막 줄이 `left.left` 인 이유는? (`left.right` 가 아니라) ⑭________________

*(답을 적은 뒤 실행)*

In [ ]:
seq3 = [9,5,10,2,7,11,1,4,6,8,12,3]
print("[⑬ is_left_child = True 초기화가 필요한 경우]")
print("  → p.left 자체가 최댓값일 때 (p.left에 오른쪽 자식이 없음)")
t = build([10, 5, 15, 3, 20])       # 5는 오른쪽 자식이 없다
show(t, "  트리: 10 삭제 예정 (왼쪽 서브트리 루트=5, 5의 오른쪽 자식 없음)")
p = t.root
left = p.left
print(f"  p={p.key}, p.left={left.key}, left.right={left.right}")
print(f"  → while left.right is not None: 이 한 번도 안 돈다!")
print(f"  → is_left_child 가 True 로 남아 parent(={p.key}).left = left.left 실행")
t.remove(10)
show(t, "  10 삭제 후")
print("  중위:", inorder(t), "✅\n")

print("[⑩ p.value 를 빠뜨리면?]")
class BugValue(BinarySearchTree):
    def remove(self, key):
        p=self.root; parent=None; il=True
        while True:
            if p is None: return False
            if key==p.key: break
            parent=p
            if key<p.key: il=True;  p=p.left
            else:         il=False; p=p.right
        if p.left is None:
            if p is self.root: self.root=p.right
            elif il: parent.left=p.right
            else: parent.right=p.right
        elif p.right is None:
            if p is self.root: self.root=p.left
            elif il: parent.left=p.left
            else: parent.right=p.left
        else:
            parent=p; left=p.left; il=True
            while left.right is not None:
                parent=left; left=left.right; il=False
            p.key = left.key
            # p.value = left.value   ← 🐛 빠뜨림
            if il: parent.left=left.left
            else:  parent.right=left.left
        return True

b = BugValue()
for k in seq3: b.add(k, k*10)
b.remove(5)
print("  버그 버전:", " | ".join(kv(b)))
a = build(seq3); a.remove(5)
print("  정상 버전:", " | ".join(kv(a)))
print("\n  🔥 '4:50' 이 나오면 value 복사를 빠뜨린 것 — 키는 맞는데 값이 틀리다!")
print("     덤프만 보면 못 잡고, key:value 를 같이 봐야 잡힌다")

---

# 💻 PART 6 — 코드 구현 (19~24번)

> 8~18번에서 손으로 다 그려봤으니 이제 코드로. **그림이 그려지면 코드는 저절로 나와.**

### 19. 🟢 [빈칸] `Node` 와 `search`

**기대 출력**
```
Node: key=5, value=50, left=None, right=None
search(3) = 30
search(8) = None
```

In [ ]:
class MyNode:
    """이진 검색 트리의 노드"""
    def __init__(self, key: Any, value: Any, left: MyNode = None, right: MyNode = None):
        self.key = ___          # ① 키
        self.value = ___        # ② 값
        self.left = ___         # ③ 왼쪽 포인터
        self.right = ___        # ④ 오른쪽 포인터


class MyBST:
    def __init__(self):
        self.root = ___         # ⑤ 빈 트리

    def search(self, key: Any) -> Any:
        p = ___                 # ⑥ 루트에 주목
        while True:
            if ___:             # ⑦ 더 진행할 수 없으면
                return None
            if key == p.key:
                return ___      # ⑧ 검색 성공 — 무엇을 반환?
            elif key < p.key:
                p = ___         # ⑨ 왼쪽 서브트리에서 검색
            else:
                p = ___         # ⑩ 오른쪽 서브트리에서 검색


n = MyNode(5, 50)
print(f"Node: key={n.key}, value={n.value}, left={n.left}, right={n.right}")

# search 테스트용으로 add를 잠시 빌려온다
MyBST.add = BinarySearchTree.add
MyBST._Node = MyNode
t = MyBST()
for k in [5,2,7,1,4,3]:
    t.add(k, k*10)
print(f"search(3) = {t.search(3)}")
print(f"search(8) = {t.search(8)}")

### 20. 🟡 [빈칸] `add`

11~12번의 알고리즘 그대로.

**기대 출력**
```
중위: [1, 2, 3, 4, 5, 6, 7]
add(5, 999) → False  (중복)
빈 트리에 add → 루트 키 = 10
```

In [ ]:
def my_add(self, key: Any, value: Any) -> bool:
    """키가 key이고 값이 value인 노드를 삽입"""

    def add_node(node: Node, key: Any, value: Any) -> bool:
        """node를 루트로 하는 서브트리에 삽입 (node는 None이 아님)"""
        if key == node.key:
            return ___                       # ① 중복 키
        elif key < node.key:
            if node.left is None:
                node.left = Node(key, value, None, None)   # ② 자리 발견!
            else:
                return ___                   # ③ 왼쪽으로 재귀
        else:
            if ___:                          # ④ 오른쪽 자리가 비었나
                node.right = Node(key, value, None, None)
            else:
                return ___                   # ⑤ 오른쪽으로 재귀
        return True

    if ___:                                  # ⑥ 트리가 비어 있으면
        self.root = Node(key, value, None, None)
        return True
    else:
        return ___                           # ⑦ 루트부터 시작


MyBST.add = my_add
t = MyBST()
for k in [5,2,7,1,4,3,6]:
    t.add(k, k*10)

def ino_my(t):
    r=[]
    def go(n):
        if n: go(n.left); r.append(n.key); go(n.right)
    go(t.root); return r

print("중위:", ino_my(t))
print("add(5, 999) →", t.add(5, 999), " (중복)")
e = MyBST(); e.add(10, 100)
print("빈 트리에 add → 루트 키 =", e.root.key)

### 21. 🔴 [빈칸] `remove` — 오늘의 최난도

13~18번에서 손으로 그린 세 경우를 전부 코드로.

**기대 출력**
```
A) 3 삭제  → [0, 1, 2, 4, 5, 6, 7, 8, 9]
B) 7 삭제  → [0, 1, 2, 3, 4, 5, 6, 8, 9]
C) 5 삭제  → [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12]
   value 확인: 4:40 ✅
D) 루트 9 삭제 → [1, 2, 3, 4, 6, 7, 8, 10, 11, 12]
없는 키 99 삭제 → False
```

In [ ]:
def my_remove(self, key: Any) -> bool:
    """키가 key인 노드를 삭제"""
    p = self.root              # 스캔 중인 노드
    parent = None              # 스캔 중인 노드의 부모
    is_left_child = ___        # ① 초기값 (18번 ⑬ 참고)

    # ── 1단계: 삭제할 키를 검색
    while True:
        if p is None:
            return ___                       # ② 그 키는 존재하지 않음
        if key == p.key:
            break                            # 검색 성공
        else:
            parent = ___                     # ③ 내려가기 전에 부모를 기록
            if key < p.key:
                is_left_child = ___          # ④
                p = p.left
            else:
                is_left_child = ___          # ⑤
                p = p.right

    # ── 2단계: 경우 A와 B (자식 0개 또는 1개)
    if p.left is None:                       # 왼쪽 자식이 없으면
        if p is self.root:
            self.root = ___                  # ⑥ 새 루트
        elif is_left_child:
            parent.left = ___                # ⑦
        else:
            parent.right = ___               # ⑧
    elif p.right is None:                    # 오른쪽 자식이 없으면
        if p is self.root:
            self.root = ___                  # ⑨
        elif is_left_child:
            parent.left = ___                # ⑩
        else:
            parent.right = ___               # ⑪

    # ── 3단계: 경우 C (자식 2개)
    else:
        parent = p
        left = ___                           # ⑫ 왼쪽 서브트리 루트에서 출발
        is_left_child = True
        while ___:                           # ⑬ 최댓값을 찾을 때까지
            parent = left
            left = ___                       # ⑭
            is_left_child = False
        p.key = ___                          # ⑮ 데이터를 옮긴다
        p.value = ___                        # ⑯ ← 빠뜨리기 쉬움! (18번)
        if is_left_child:
            parent.left = ___                # ⑰ (left.right가 아니다!)
        else:
            parent.right = ___               # ⑱
    return True


MyBST.remove = my_remove

def kv_my(t):
    r=[]
    def go(n):
        if n: go(n.left); r.append(f"{n.key}:{n.value}"); go(n.right)
    go(t.root); return r
def build_my(seq):
    t = MyBST()
    for k in seq: t.add(k, k*10)
    return t

s2 = [6,2,7,1,4,8,3,5,9,0]
s3 = [9,5,10,2,7,11,1,4,6,8,12,3]

t = build_my(s2); t.remove(3); print("A) 3 삭제  →", ino_my(t))
t = build_my(s2); t.remove(7); print("B) 7 삭제  →", ino_my(t))
t = build_my(s3); t.remove(5); print("C) 5 삭제  →", ino_my(t))
print("   value 확인:", [x for x in kv_my(t) if x.startswith('4:')][0],
      "✅" if "4:40" in kv_my(t) else "❌")
t.remove(9); print("D) 루트 9 삭제 →", ino_my(t))
print("없는 키 99 삭제 →", t.remove(99))

### 22. 🟢 [빈칸] `dump` — 중위 순회의 실체

32일차에서 배운 중위 순회가 그대로 나와.

**기대 출력**
```
오름차순: 1 2 3 4 5 6 7
내림차순: 7 6 5 4 3 2 1
```

In [ ]:
def my_dump(self, reverse=False) -> None:
    """모든 노드를 키의 오름차순(또는 내림차순)으로 출력"""

    def print_subtree(node: Node):
        """오름차순 — 32일차 18번의 중위 순회"""
        if node is not None:
            print_subtree(___)               # ① 왼쪽 서브트리
            print(f"{node.key} ", end="")    # ② 노드 방문
            print_subtree(___)               # ③ 오른쪽 서브트리

    def print_subtree_rev(node: Node):
        """내림차순 — 좌우를 뒤집는다"""
        if node is not None:
            print_subtree_rev(___)           # ④
            print(f"{node.key} ", end="")
            print_subtree_rev(___)           # ⑤

    print_subtree_rev(self.root) if reverse else print_subtree(self.root)


MyBST.dump = my_dump
t = build_my([4,2,6,1,3,5,7])
print("오름차순: ", end="")
t.dump()
print()
print("내림차순: ", end="")
t.dump(reverse=True)
print()

### 23. 🟡 [빈칸] `min_key` / `max_key`

**기대 출력**
```
최솟값 = 1, 최댓값 = 7
빈 트리: 최솟값 = None, 최댓값 = None
```

In [ ]:
def my_min_key(self) -> Any:
    """가장 작은 키"""
    if ___:                      # ① 빈 트리면
        return None
    p = self.root
    while ___:                   # ② 어느 방향으로 끝까지?
        p = ___                  # ③
    return ___                   # ④ ← 이 줄을 빠뜨리면 None이 나온다!


def my_max_key(self) -> Any:
    """가장 큰 키"""
    if self.root is None:
        return None
    p = self.root
    while ___:                   # ⑤
        p = ___                  # ⑥
    return p.key


MyBST.min_key = my_min_key
MyBST.max_key = my_max_key
t = build_my([4,2,6,1,3,5,7])
print(f"최솟값 = {t.min_key()}, 최댓값 = {t.max_key()}")
e = MyBST()
print(f"빈 트리: 최솟값 = {e.min_key()}, 최댓값 = {e.max_key()}")

### 24. 🔴 [디버깅] 🔥 실제로 겪은 버그

`bst.py` 를 만들 때 `min_key` 가 계속 `None` 을 반환하는 일이 있었지. 그때 코드는 이랬어:

```python
def min_key(self) -> Any:
    """가장 작은 키"""
    if self.root is not None:      # 🐛
        return None
    p = self.root
    while p.left is not None:
        p = p.left
    return p.key
```

- 무엇이 잘못됐지? ①________________
- 트리에 노드가 **있을 때** 이 함수는 어디서 리턴해? ②____
- 트리가 **비었을 때**는? `p = self.root` 가 `None` 인데 `p.left` 를 읽으면? ③________________
- 🔥 그런데 **`max_key` 는 멀쩡했어.** 두 함수가 완전 대칭인데 왜 하나만 틀렸을까? → 복붙 후 수정하다 생긴 전형적인 실수야.

**같은 증상(`None` 반환)을 내는 다른 원인 두 가지도 찾아봐:**
- 원인 B: ④________________
- 원인 C: ⑤________________

💡 **교훈**: 파이썬은 `return` 이 없으면 **에러 없이 조용히 `None`** 을 반환해. 그래서 이런 버그는 **실행은 되는데 값만 이상한** 형태로 나타나. 22일차 10번, 29일차 20번과 같은 계열이야.

*(답을 적은 뒤 실행)*

In [ ]:
t = build([5,2,7,1,4,3])

def min_bug_A(self):
    if self.root is not None: return None       # 🐛 조건 반전
    p = self.root
    while p.left is not None: p = p.left
    return p.key

def min_bug_B(self):
    if self.root is None: return None
    p = self.root
    while p.left is not None: p = p.left
    # return p.key                              # 🐛 리턴 누락

def min_bug_C(self):
    if self.root is None:
        return None
        p = self.root                           # 🐛 if 블록 안으로 들여쓰기
        while p.left is not None: p = p.left
        return p.key

def min_bug_D(self):
    if self.root is None: return None
    p = self.root
    while p.left is not None:
        p = p.left
        return p.key                            # 🐛 while 안에서 리턴

print("정답:", t.min_key())
for nm, fn in [("A) is not None 반전", min_bug_A), ("B) return 누락", min_bug_B),
               ("C) if 안 들여쓰기", min_bug_C), ("D) while 안 return", min_bug_D)]:
    print(f"  {nm:22s} → {fn(t)}")
print("\n→ A, B, C 는 None. D는 2가 나온다 (증상으로 원인을 좁힐 수 있다)")

print("\n[빈 트리에서 A 버전은 어떻게 되나?]")
e = BinarySearchTree()
try:
    min_bug_A(e)
    print("  에러 없음 (첫 줄에서 return None 안 타고, root가 None이라 조건 거짓)")
    print("  → p = None 이고 p.left 접근 →", end=' ')
except AttributeError as ex:
    print(f"  AttributeError: {ex}")
    print("  🔥 빈 트리에서는 크래시까지 난다!")

---

# 📐 PART 7 — 성능과 균형 (25~27번)

### 25. 🔴 [실험] 🔥 삽입 순서가 성능을 결정한다

교재 383p 보충수업 9-1:
> **"다음에 다룰 이진 검색 트리는 키의 오름차순으로 노드가 삽입되면 트리의 높이가 깊어지는 단점이 있습니다. 예를 들어 비어 있는 이진 검색 트리에 1, 2, 3, 4, 5 순으로 노드를 삽입하면 [그림 9C-1]처럼 직선 모양의 트리가 됩니다(실제로 선형 리스트처럼 되어 아주 빠른 검색을 수행할 수 없습니다)."**

**예측해봐** (n = 100,000):
- `1, 2, 3, ..., 100000` 순서로 삽입 → 높이 = ①____
- **랜덤 순서**로 삽입 → 높이 = ②____
- `log₂(100000)` = ③____

그리고 **최악 키를 검색할 때 비교 횟수**는?
- 오름차순 트리에서: ④____
- 랜덤 트리에서: ⑤____

*(예측을 적은 뒤 실행)*

In [ ]:
class FastNode:
    __slots__=('k','l','r')
    def __init__(s,k): s.k=k; s.l=None; s.r=None
def add_it(root,k):
    if root is None: return FastNode(k)
    p=root
    while True:
        if k<p.k:
            if p.l is None: p.l=FastNode(k); return root
            p=p.l
        elif k>p.k:
            if p.r is None: p.r=FastNode(k); return root
            p=p.r
        else: return root
def build_fast(seq):
    r=None
    for k in seq: r=add_it(r,k)
    return r
def height_it(t):
    if t is None: return -1
    st=[(t,0)]; mx=0
    while st:
        n,d=st.pop(); mx=max(mx,d)
        if n.l: st.append((n.l,d+1))
        if n.r: st.append((n.r,d+1))
    return mx
def cmps(t,k):
    c=0
    while t:
        c+=1
        if k==t.k: return c
        t=t.l if k<t.k else t.r
    return c

print("     n | 오름차순 삽입 높이 | 랜덤 삽입 높이 | log2(n)")
print("-"*58)
for n in (15, 1000, 10000, 100000):
    asc=list(range(1,n+1))
    random.seed(0); rnd=asc[:]; random.shuffle(rnd)
    ta=build_fast(asc); tr=build_fast(rnd)
    print(f" {n:6d} | {height_it(ta):17d} | {height_it(tr):14d} | {math.log2(n):6.1f}")

print("\n[최악 키 검색 비교 횟수, n = 10,000]")
n=10000; asc=list(range(1,n+1))
random.seed(0); rnd=asc[:]; random.shuffle(rnd)
ta=build_fast(asc); tr=build_fast(rnd)
print(f"  오름차순 트리에서 {n} 검색: {cmps(ta,n):,}번 🔥")
print(f"  랜덤    트리에서 {n} 검색: {cmps(tr,n):,}번")
print(f"  → {cmps(ta,n)//cmps(tr,n):,}배 차이!")

print("\n🔥 오름차순 삽입 = 사실상 연결 리스트")
print("   트리를 쓴 의미가 완전히 사라진다 (O(log n) → O(n))")

### 26. 🟡 [설명] 왜 이런 일이 생길까 — 21일차와의 연결

25번의 결과를 **21일차 퀵 정렬**과 나란히 놓아봐.

| | 나쁜 선택 | 결과 |
|---|---|---|
| 퀵 정렬 (21일차) | ① ________ 피벗 + 정렬된 입력 | O(n²) |
| BST (오늘) | ② ________ | 높이 O(n) → 검색 O(n) |

- 🔥 **공통 원인**을 한 문장으로: ③________________
- 21일차에서 퀵 정렬은 어떻게 해결했지? ④________________ (22일차 `sort3`)
- BST는 왜 같은 방법을 못 쓸까? ⑤________________
  💡 힌트: 퀵 정렬은 데이터를 **전부 알고** 시작하지만, BST는 값이 **하나씩 들어와**.

**루트가 중요한 이유를 확인해봐:**
같은 12개 키를 넣는데 **첫 삽입만** 바꾸면?

| 첫 삽입 키 | 왼쪽에 갈 개수 | 오른쪽에 갈 개수 |
|---|---|---|
| 1 | ⑥ | ⑦ |
| 6 | ⑧ | ⑨ |
| 12 | ⑩ | ⑪ |

- **가장 균형 잡힌 루트**는? ⑫____ 그건 전체의 ⑬________ 값이지.

*(답을 적은 뒤 실행)*

In [ ]:
class FastNode:
    __slots__=('k','l','r')
    def __init__(s,k): s.k=k; s.l=None; s.r=None
def add_it(root,k):
    if root is None: return FastNode(k)
    p=root
    while True:
        if k<p.k:
            if p.l is None: p.l=FastNode(k); return root
            p=p.l
        elif k>p.k:
            if p.r is None: p.r=FastNode(k); return root
            p=p.r
        else: return root
def build_fast(seq):
    r=None
    for k in seq: r=add_it(r,k)
    return r
def height_it(t):
    if t is None: return -1
    st=[(t,0)]; mx=0
    while st:
        nd,d=st.pop(); mx=max(mx,d)
        if nd.l: st.append((nd.l,d+1))
        if nd.r: st.append((nd.r,d+1))
    return mx

keys = list(range(1, 13))
print("같은 12개 키, 첫 삽입만 다르게:\n")
print("  첫 삽입 | 왼쪽 | 오른쪽 | 최종 높이(나머지는 오름차순)")
print("  " + "-"*52)
for first in (1, 3, 6, 9, 12):
    rest = [k for k in keys if k != first]
    t = build_fast([first] + rest)
    left_n  = sum(1 for k in keys if k < first)
    right_n = sum(1 for k in keys if k > first)
    print(f"    {first:2d}    | {left_n:4d} | {right_n:6d} | {height_it(t):3d}")

print("\n→ 6 또는 7(중앙값)이 루트일 때 좌우가 가장 고르게 갈린다")
print("→ 21일차 퀵 정렬의 '피벗 선택'과 정확히 같은 문제 🔥")

print("\n[랜덤 삽입이면 평균적으로 괜찮다]")
random.seed(1)
hs = []
for _ in range(200):
    s = list(range(1, 1001)); random.shuffle(s)
    hs.append(height_it(build_fast(s)))
print(f"  n=1000, 랜덤 삽입 200회 → 높이 평균 {sum(hs)/len(hs):.1f}, "
      f"최소 {min(hs)}, 최대 {max(hs)}")
print(f"  log2(1000) = {math.log2(1000):.1f} → 약 2배 수준으로 유지됨 ✅")
print("  💡 이론적으로 랜덤 삽입 BST의 평균 높이는 약 4.3 log₂n 이다")

### 27. 🟡 [설명] 균형 검색 트리 — 교재 보충수업 9-1

교재 383p:
> **"이와 같이 높이를 O(log n)으로 제한하여 고안한 검색 트리를 균형 검색 트리(self-balancing search tree)라고 합니다."**
>
> 이진 균형 검색 트리: **AVL 트리**, **레드·블랙 트리**
> 이진이 아닌 균형 검색 트리: **B 트리**, **2-3 트리**

**표를 채워봐:**

| | 자식 수 | 균형 유지 방법 | 특징 | 대표 사용처 |
|---|---|---|---|---|
| **AVL 트리** | 2 | 높이차 ≤ 1, ①____ | 검색이 가장 빠름 | 검색 위주 |
| **레드·블랙 트리** | 2 | ②____ 규칙, 회전 | ③________ | ④________________ |
| **2-3 트리** | ⑤____ | 노드 분할/병합 | 회전 불필요 | 교육용, 레드·블랙의 기반 |
| **B 트리** | ⑥____ | 노드 분할/병합 | ⑦________________ | ⑧________________ |

**핵심 질문 3개**
1. AVL과 레드·블랙은 **무엇을 맞바꿨을까?** ⑨________________
2. B 트리는 왜 **한 노드에 키를 여러 개** 담을까? ⑩________________
   💡 힌트: 디스크는 한 번 읽을 때 **블록 단위**로 읽어서, 키 1개를 읽든 100개를 읽든 비용이 같아.
3. 교재의 `BinarySearchTree` 는 균형을 맞춰줄까? ⑪____
   → 그래서 실무에서 정렬된 맵이 필요하면 대개 ⑫________________ 를 쓰지.

*(답을 적은 뒤 실행)*

In [ ]:
print("[B 트리가 디스크에 유리한 이유 — 계산으로 확인]")
n = 1_000_000
print(f"  데이터 {n:,}개를 저장할 때\n")
print(f"  이진 트리(자식 2개):  높이 ≈ log₂({n:,}) = {math.log2(n):.0f}")
print(f"    → 디스크 접근 {math.log2(n):.0f}번\n")
for m in (10, 100, 1000):
    h = math.log(n, m)
    print(f"  B 트리(자식 {m:4d}개): 높이 ≈ log_{m}({n:,}) = {h:.1f}")
    print(f"    → 디스크 접근 약 {math.ceil(h)}번")
print("\n🔥 자식이 100개면 100만 개도 3번이면 도달")
print("   디스크는 한 번 읽는 비용이 크니 '한 번에 많이'가 압도적으로 유리\n")

print("[파이썬에는 균형 트리가 표준으로 없다]")
print("  · C++  → std::map (레드·블랙)")
print("  · Java → TreeMap (레드·블랙)")
print("  · 파이썬 → 표준 라이브러리에 없음")
print("     대안 1: bisect + 정렬 리스트 (검색 O(log n), 삽입 O(n))")
print("     대안 2: sortedcontainers 패키지 (외부)")
print("     대안 3: dict (순서 없음, 검색 O(1)) ← 대부분 이걸로 충분")

import bisect
a = []
for k in [5,2,7,1,4]:
    bisect.insort(a, k)
print(f"\n  bisect 예시: {a}  ← 항상 정렬 유지")
print(f"  4의 위치: {bisect.bisect_left(a, 4)} (검색은 O(log n), 삽입은 O(n))")

---
---

# ✅ 정답 & 해설

> ⚠️ **트리를 종이에 그리고 손으로 푼 뒤에 내려와.** 특히 13~17번은 직접 그려야 남아.

---

## 🔁 Remind

### R-1
- ① `20 → 30 → 40 → 50 → 60 → 70 → 80`
- ② **오름차순으로 정렬되어 나온다**
- ③ **`dump()`** — 오늘 22번에서 구현해

### R-2
- ① **절반** ② **O(log n)**
- ③ **주목 노드 `p` 자신** — 배열은 인덱스를 계산해서 가운데를 찾았지만, 트리는 **구조 자체가 이미 가운데를 가리키고 있어**
- ④ 🔥 **삽입이 O(log n)** 이야. 정렬된 배열은 한가운데에 값을 넣으려면 뒤의 원소를 전부 밀어야 해서 **O(n)** 이거든. BST는 리프 자리에 붙이기만 하면 돼 (29일차 1번과 같은 이야기).

---

## 🌲 PART 1 해설

### 1. 이진 트리
- ① **모든 노드의 자식이 왼쪽·오른쪽 최대 2개**
- ② **모든 노드의 차수가 2 이하**
- ③ **속해.** 자식이 0개(리프)든 1개든 2개든 전부 이진 트리야. 32일차 3번 (나)의 "왼쪽만 있는 D"를 떠올려봐.
- ④ **순서 트리** ⑤ **성립 못 해.** BST는 "왼쪽 = 작은 값, 오른쪽 = 큰 값"이 규칙 자체인데, 좌우를 구분 안 하면 그 규칙을 쓸 수 없어.

### 2. 왼쪽/오른쪽 서브트리
- ① `{4, 1}` ② `{7, 6, 9}` ③ `{13, 12, 14}`
- ④ **6개** (5,4,7,1,6,9) ⑤ **5개** (15,13,18,12,14)
- ⑥ **둘 다 빈 트리(`None`)** — 32일차 10번의 빈 트리 개념

### 3. 완전 이진 트리

| | 완전? | 이유 |
|---|---|---|
| (가) | ① **✅** | ② 마지막 레벨을 왼쪽부터 채움 |
| (나) | ③ **❌** | ④ 마지막 레벨에서 D 다음이 비고 E가 떨어져 있음 |
| (다) | ⑤ **✅** | ⑥ 마지막 레벨 D, E, F 를 왼쪽부터 빈틈없이 채움 |
| (라) | ⑦ **✅** | ⑧ 왼쪽 자식부터 채웠음 |
| (마) | ⑨ **❌** | ⑩ 왼쪽을 건너뛰고 오른쪽부터 채움 |

- ⑪ **둘 다 "왼쪽부터 빈틈없이"를 어겼어.** 왼쪽 자리를 비워둔 채 오른쪽에 노드가 있으면 완전 이진 트리가 아니야.

### 4. 완전 이진 트리와 배열
- ① `(i-1)//2` ② `2i+1` ③ `2i+2`
- ④ **너비 우선 순회 = 레벨 순서 = 왼쪽부터** 인데, 배열 인덱스도 0,1,2,... 로 같은 순서로 붙기 때문
- ⑤ **중간에 빈 자리가 생기면 인덱스가 밀려서** `2i+1` 공식이 깨져. 빈 자리를 `None` 으로 채우면 공식은 유지되지만 **메모리가 기하급수로 낭비**돼 (29일차 5번)
- ⑥ `2^(k+1) - 1` ⑦ `log₂(n+1) - 1 ≈ O(log n)`
- ⑧ 🔥 유도:
```
n = 2^(k+1) - 1
n + 1 = 2^(k+1)
log₂(n+1) = k + 1
k = log₂(n+1) - 1
```

**실측**: 높이 30이면 노드 **21억 개**. 뒤집으면 21억 개짜리 트리도 **깊이 30**이면 도달해. `k` 가 1 늘 때마다 `n` 이 2배가 되니, 거꾸로 `n` 이 2배가 돼도 `k` 는 1만 늘지.

---

## 🔑 PART 2 해설

### 5. BST 조건
- ① **같은 키가 두 개 있으면 어느 쪽으로 내려가야 할지 정할 수 없어.** 왼쪽 조건(`<`)에도 오른쪽 조건(`>`)에도 안 맞거든. 그래서 `add` 가 `key == node.key` 일 때 `False` 를 반환해.
- ② `{5, 4, 7, 1, 6, 9}` — 전부 11보다 작음 ✅

### 6. 🔥 BST 판별

| | BST? | 위반 |
|---|---|---|
| (가) | ① **✅** | ② 없음 |
| (나) | ③ **❌** | ④ **8 ↔ 9** (9가 8의 왼쪽 서브트리에 있음) |
| (다) | ⑤ **✅** | ⑥ 없음 |
| (라) | ⑦ **❌** | ⑧ **5 ↔ 6** (6이 5의 왼쪽 서브트리에 있음) |

**실측**
```
       자식만 검사      범위 검사(정답)   중위 순회
(나)      True            False        [1, 3, 4, 6, 9, 8, 10]  ← 자식만 보면 속는다! 🔥
(라)      True            False        [3, 6, 5, 7, 9]         ← 자식만 보면 속는다! 🔥
```

- ⑨ **(나)의 `9` 는 부모 `6` 보다 크니 오른쪽 자식으로 OK처럼 보이지만, 조상 `8` 의 왼쪽 서브트리에 있어서 위반.**
  **(라)의 `6` 도 부모 `3` 보다 커서 OK 같지만, 조상 `5` 의 왼쪽 서브트리에 있어 위반.**
- ⑩ `3 < 9 < 8` 을 만족해야 했는데 불가능 → 애초에 그 자리에 올 수 없는 값

> 🔑 **BST 검증의 정석**: 위에서 아래로 **[하한, 상한] 범위를 물려주며** 검사하거나, **중위 순회가 오름차순인지** 확인하면 돼. 후자가 더 간단하지.

### 7. BST의 네 가지 특징
- ⓐ **중위 순회가 "왼쪽 → 자신 → 오른쪽"인데, BST에서 왼쪽은 항상 작고 오른쪽은 항상 크기 때문**
- ⓑ **한 번 비교할 때마다 한쪽 서브트리를 통째로 버리므로 후보가 절반씩 줄어듦** (균형 잡혔을 때)
- ⓒ **삽입 위치가 항상 리프 자리라서, 검색해서 내려간 뒤 포인터 하나만 연결하면 끝**
- ⓓ **일치 ✅**
- ⓔ **뒤의 원소를 전부 한 칸씩 밀어야 해서 O(n)** — 29일차 1번

---

## 🔍 PART 3 해설

### 8. 검색 성공

| 단계 | p | 비교 | 다음 |
|---|---|---|---|
| 1 | 5 | 3 < 5 | ① **왼쪽으로** |
| 2 | ② **2** | ③ **3 > 2** | ④ **오른쪽으로** |
| 3 | ⑤ **4** | ⑥ **3 < 4** | ⑦ **왼쪽으로** |
| 4 | ⑧ **3** | ⑨ **3 == 3** | ⑩ **성공, value 반환** |

- ⑪ **4번**
- ⑫ **3** ⑬ **레벨 0부터 세니까.** 높이 3이면 레벨이 0,1,2,3 네 개고 각 레벨에서 한 번씩 비교하니 최대 4번.

### 9. 검색 실패

| 단계 | p | 비교 | 다음 |
|---|---|---|---|
| 1 | 5 | 8 > 5 | ① **오른쪽으로** |
| 2 | ② **7** | ③ **8 > 7** | ④ **p = None** |
| 3 | ⑤ **None** | - | ⑥ **실패, None 반환** |

- ⑦ `if p is None: return None`

**실측**

| 키 | 경로 | 결과 |
|---|---|---|
| 1 | ⑧ `5 → 2 → 1` | ⑨ 성공 |
| 4 | ⑩ `5 → 2 → 4` | ⑪ 성공 |
| 6 | ⑫ `5 → 7` | ⑬ 실패 |
| 0 | ⑭ `5 → 2 → 1` | ⑮ 실패 |

🔥 **`6` 과 `0` 의 경로가 짧지?** 트리에 없는 값은 **빨리 포기**할 수 있어. 반면 배열 선형 검색은 끝까지 훑어야 하고.

### 10. 검색 알고리즘
- ① **None** ② **검색 성공, 종료** ③ **왼쪽 자식 노드로 옮김** ④ **오른쪽 자식 노드로 옮김** ⑤ **2**
- ⑥ **이진 검색은 `pl`/`pr` 로 범위를 계산해서 가운데를 구하지만, BST는 노드가 이미 "가운데"를 담고 있어서 계산이 필요 없어.**
- ⑦ **절반** — 왼쪽으로 가면 오른쪽 서브트리 전체를 버리는 셈이니까

---

## ➕ PART 4 해설

### 11. 삽입
- ① `2` (6 → 왼쪽 2 → 2의 왼쪽 비었음)
- ② `2` ③ `4` (6 → 왼쪽 2 → 오른쪽 4 → 4의 오른쪽 비었음)
- ④ `1 → 2 → 4 → 5 → 6 → 7`
- ⑤ **삽입 위치를 "검색"으로 찾는데, 검색이 실패하는 지점(= `None` 을 만나는 자리)이 곧 빈 리프 자리이기 때문**
- ⑥ **없어** ✅ 화살표 하나만 연결하면 끝. 이게 BST의 큰 장점이야.

### 12. 삽입 알고리즘
- ① **삽입을 실패하고 종료 (`False` 반환)**
- ② **그 자리에 노드를 삽입하고 종료** ③ **주목 노드를 왼쪽 자식으로 옮김(재귀 호출)**
- ④ **그 자리에 삽입하고 종료** ⑤ **오른쪽 자식으로 옮김(재귀 호출)**
- ⑥ **BST 조건상 같은 키가 두 개 있으면 어느 쪽으로 갈지 정할 수 없어서** (5번)
- ⑦ **트리가 비어 있을 때(루트가 `None`)**
- ⑧ 🔥 **`add_node` 안에서 `node.key` 를 읽으려면 `node` 가 `None` 이 아니어야 해.** 루트가 `None` 인 경우만 바깥에서 걸러내면, 내부 함수는 그 걱정 없이 단순해져. 29일차 8번(`add_last` 가 빈 리스트를 `add_first` 로 넘긴 것)과 같은 설계야.

---

## ➖ PART 5 해설

### 트리 ②의 차수

| 노드 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| 차수 | 0 | 1 | 2 | 0 | 2 | 0 | 2 | 1 | 1 | 0 |

- ① 차수 0: **0, 3, 5, 9** ② 차수 1: **1, 7, 8** ③ 차수 2: **2, 4, 6**

### 13. 경우 A — 자식 0개
**ⓐ `3` 삭제**
- ① `6 → 2 → 4 → 3` ② **4** ③ **왼쪽** ④ `4.left = None`
- ⑤ `[0, 1, 2, 4, 5, 6, 7, 8, 9]`

**ⓑ `9` 삭제**
- ⑥ `6 → 7 → 8 → 9` ⑦ **8** ⑧ **오른쪽** ⑨ `8.right = None`
- ⑩ `[0, 1, 2, 3, 4, 5, 6, 7, 8]`

- ⑪ **29일차 9번과 똑같아**: "삭제"란 노드 객체를 지우는 게 아니라 **가리키는 화살표를 끊는 것**이야. 아무도 안 가리키면 파이썬이 알아서 회수해.

### 14. 경우 B — 자식 1개
**ⓐ `7` 삭제**
- ① **6** ② **오른쪽** ③ **8** ④ `6.right = 8`
- ⑤ `[0, 1, 2, 3, 4, 5, 6, 8, 9]`
- ⑥ 🔥 **`8` 을 루트로 하는 서브트리 전체(8, 9)가 `6` 보다 크다**는 게 보장돼야 `6.right` 에 붙여도 BST가 유지돼. 교재 391p가 이걸 짚은 거야.

**ⓑ `1` 삭제**
- ⑦ **2** ⑧ **왼쪽** ⑨ `2.left = 0` ⑩ `[0, 2, 3, 4, 5, 6, 7, 8, 9]`

**A와 B를 한 코드로**
- ⑪ **첫 번째 분기 (`if p.left is None`)** ⑫ **`p.right`, 즉 `None`**
- → 부모의 포인터에 `None` 을 넣게 되어 **경우 A와 완전히 동일한 동작** ✅

> 🔑 **`None` 을 "빈 트리"로 정의한 덕분에 두 경우가 하나로 합쳐졌어.** 29일차 7번, 31일차 7번의 "더미가 분기를 없앤다"와 같은 발상이야.

### 15. 🔥 경우 C — 자식 2개
- ① **2** ② `{2, 1, 4, 3}` ③ **4**
- ④ **왼쪽 서브트리 루트에서 `right` 를 따라 끝까지 간다** (32일차 25번: 최댓값은 가장 오른쪽 끝)
- ⑤ **4**
- ⑥ 🔥 **노드를 옮기면 그 노드를 가리키던 부모의 포인터, 그 노드의 두 자식 포인터를 전부 다시 연결해야 해서 복잡해.** 데이터(key, value)만 복사하면 **포인터는 하나도 안 건드려도** 되지.
- ⑦ **왼쪽 자식 `3` 만 있음** ⑧ **B** ⑨ **2** ⑩ **오른쪽** ⑪ `2.right = 3`
- ⑫ **4** ⑬ **3**
- ⑭ `[1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12]`
- ⑮ **40** — `p.value = left.value` 를 빠뜨리면 `4:50` 이 나와 (18번)

### 16. 왜 "왼쪽 서브트리의 최댓값"인가
- ① **2개** (선행자와 후행자)
- ② **바로 앞 (선행자, predecessor)** ③ **바로 뒤 (후행자, successor)**
- ④ **아무거나 써도 돼!** 실측에서 확인했지 — 교재 방식(4가 올라옴)과 변형(6이 올라옴) 모두 중위 순회 결과가 같은 유효한 BST야. 모양만 다를 뿐.
- ⑤ **4** ⑥ **6**
- ⑦ 🔥 **최댓값은 정의상 "오른쪽으로 갈 수 있는 데까지 간 노드"라, 오른쪽 자식이 없어야만 최댓값이야.**
  → 그래서 그 노드를 삭제할 때는 **자식이 0개 또는 1개(왼쪽만)** 뿐이라 **경우 C가 재귀적으로 반복되지 않아.** 이게 알고리즘이 반드시 끝나는 이유야.

### 17. 루트 삭제
- ① **2** ② **C** ③ **4** ④ **8**
- ⑤ **없음(리프)** ⑥ **A** ⑦ **7** ⑧ **오른쪽**
- ⑨ **8** (노드 객체는 그대로고 key/value만 8/80으로 바뀜)
- ⑩ `[1, 2, 3, 4, 6, 7, 8, 10, 11, 12]`
- ⑪ **부모가 없어서** `parent.left/right` 를 갱신할 수가 없어. 그래서 `self.root` 자체를 바꿔야 해.
- ⑫ 🔥 **경우 C는 노드를 제거하는 게 아니라 데이터만 덮어쓰기 때문.** 루트 객체는 그대로 남고 `key`/`value`만 교체되니 `self.root` 를 건드릴 필요가 없어. 실제로 지워지는 건 **최댓값 노드**인데 그건 절대 루트가 아니지.

### 18. 삭제 알고리즘 정리
**1단계**
- ① **삭제할 노드** ② **그 부모 노드**
- ③ **부모의 어느 쪽 포인터를 갱신할지 알아야 하니까.** 왼쪽 자식이면 `parent.left`, 오른쪽이면 `parent.right` 를 고쳐야 해.
- ④ **`parent` 와 `is_left_child` 를 기록하는 것.** 검색만 할 때는 필요 없었지.

**2단계**
- ⑤ **자식이 0개 + 왼쪽만 없는 경우(=오른쪽 자식만 있음)**
- ⑥ **오른쪽만 없는 경우(=왼쪽 자식만 있음)**
- ⑦ **삭제할 노드가 루트인가 / 부모의 왼쪽 자식인가 / 오른쪽 자식인가**

**3단계**
- ⑧ **왼쪽 서브트리의 최댓값** ⑨ **키를 옮김** ⑩ **값을 옮김 — 빠뜨리면 키는 맞는데 값이 틀림**
- ⑪ **최댓값 노드** ⑫ **그 부모**
- ⑬ 🔥 **`while` 이 한 번도 안 도는 경우가 있어서.** `p.left` 자체가 최댓값일 때(= `p.left.right` 가 `None`), `parent` 는 `p` 그대로고 `left` 는 `p.left` 야. 이때 `p.left = left.left` 를 해야 하니 `is_left_child` 가 `True` 여야 해. 실측에서 `[10,5,15,3,20]` 트리로 확인했지.
- ⑭ **최댓값 노드는 오른쪽 자식이 없으니 남은 건 왼쪽 자식뿐**이라서 (16번 ⑦)

---

## 💻 PART 6 해설

### 19~23. 빈칸 정답

**19번**
```python
self.key = key; self.value = value; self.left = left; self.right = right   # ①②③④
self.root = None                              # ⑤
p = self.root                                 # ⑥
if p is None:                                 # ⑦
return p.value                                # ⑧
p = p.left                                    # ⑨
p = p.right                                   # ⑩
```

**20번**
```python
return False                                  # ①
node.left = Node(key, value, None, None)      # ②
return add_node(node.left, key, value)        # ③
if node.right is None:                        # ④
return add_node(node.right, key, value)       # ⑤
if self.root is None:                         # ⑥
return add_node(self.root, key, value)        # ⑦
```

**21번**
```python
is_left_child = True                          # ①
return False                                  # ②
parent = p                                    # ③
is_left_child = True                          # ④
is_left_child = False                         # ⑤
self.root = p.right                           # ⑥
parent.left = p.right                         # ⑦
parent.right = p.right                        # ⑧
self.root = p.left                            # ⑨
parent.left = p.left                          # ⑩
parent.right = p.left                         # ⑪
left = p.left                                 # ⑫
while left.right is not None:                 # ⑬
left = left.right                             # ⑭
p.key = left.key                              # ⑮
p.value = left.value                          # ⑯  ← 빠뜨리기 쉬움!
parent.left = left.left                       # ⑰
parent.right = left.left                      # ⑱
```

**22번**
```python
print_subtree(node.left)   # ①
print_subtree(node.right)  # ③
print_subtree_rev(node.right)  # ④
print_subtree_rev(node.left)   # ⑤
```
🔥 **내림차순은 좌우만 뒤집으면 끝.** 32일차 19번의 "세 순회는 같은 경로, 기록 시점만 다르다"의 응용이야.

**23번**
```python
if self.root is None:      # ①
while p.left is not None:  # ②
    p = p.left             # ③
return p.key               # ④  ← 이 줄!
while p.right is not None: # ⑤
    p = p.right            # ⑥
```

### 24. 🔥 실제로 겪은 버그
- ① **조건이 `is None` 이 아니라 `is not None` 으로 반전됨**
- ② **첫 줄에서 바로 `return None`** — 트리에 뭐가 있든 무조건 `None`
- ③ **`AttributeError: 'NoneType' object has no attribute 'left'`** — 빈 트리에서는 크래시까지 나
- ④ 원인 B: **`return p.key` 를 빠뜨림** (파이썬은 조용히 `None` 반환)
- ⑤ 원인 C: **함수 본문이 `if` 블록 안으로 들여쓰기됨** (도달 불가 코드)

**실측**
```
정답:                   1
A) is not None 반전  → None
B) return 누락       → None
C) if 안 들여쓰기    → None
D) while 안 return   → 2      ← 증상이 다르니 원인을 좁힐 수 있다
```

> 🔑 **증상으로 원인을 좁혀라.** `None` 이면 A/B/C, `2` 면 D. 그리고 `max_key` 가 멀쩡한데 `min_key` 만 틀렸다면 **두 함수를 나란히 놓고 대조**하는 게 가장 빨라.

---

## 📐 PART 7 해설

### 25. 🔥 삽입 순서가 성능을 결정

**실측**
```
     n | 오름차순 삽입 높이 | 랜덤 삽입 높이 | log2(n)
    15 |                14 |             6 |    3.9
  1000 |               999 |            20 |   10.0
 10000 |              9999 |            29 |   13.3
100000 |             99999 |            38 |   16.6
```

- ① **99,999** ② **38** ③ **16.6**
- ④ **10,000번** ⑤ **6번** → 🔥 **1,666배 차이**

**오름차순 삽입 = 사실상 연결 리스트.** 트리를 쓴 의미가 완전히 사라져.

### 26. 21일차와의 연결

| | 나쁜 선택 | 결과 |
|---|---|---|
| 퀵 정렬 | ① **맨 앞/맨 뒤** 피벗 + 정렬된 입력 | O(n²) |
| BST | ② **정렬된 순서로 삽입** (루트가 최솟값/최댓값이 됨) | 높이 O(n) |

- ③ 🔥 **분할이 한쪽으로 완전히 치우쳐서, "절반씩 줄인다"는 전제가 깨지기 때문**
- ④ **`sort3` 로 세 값의 중앙값을 피벗으로 선택** (22일차 9번)
- ⑤ **퀵 정렬은 데이터를 전부 알고 시작하지만, BST는 값이 하나씩 들어와서 미래를 모르기 때문.** 지금 넣는 값이 나중에 중앙값이 될지 알 수가 없어. 그래서 **삽입 후에 모양을 고치는**(회전) 방식으로 해결할 수밖에 없고, 그게 균형 검색 트리야.

**실측 (첫 삽입만 다르게)**
```
  첫 삽입 | 왼쪽 | 오른쪽 | 높이
      1   |    0 |     11 |  11
      3   |    2 |      9 |   9
      6   |    5 |      6 |   6
      9   |    8 |      3 |   8
     12   |   11 |      0 |  11
```
- ⑥ 0 ⑦ 11 / ⑧ 5 ⑨ 6 / ⑩ 11 ⑪ 0
- ⑫ **6 (또는 7)** ⑬ **중앙값**

💡 **랜덤 삽입이면 평균적으로 괜찮아.** n=1000에서 랜덤 삽입 200회의 평균 높이가 약 20으로, `log₂(1000)=10` 의 약 2배 수준이야. 이론적으로 랜덤 BST의 평균 높이는 약 `4.3 log₂n` 으로 알려져 있어.

### 27. 균형 검색 트리

| | 자식 | 균형 방법 | 특징 | 사용처 |
|---|---|---|---|---|
| AVL | 2 | ① **회전(rotation)** | 검색 최고속 | 검색 위주 |
| 레드·블랙 | 2 | ② **색(빨강/검정)** 규칙, 회전 | ③ **삽입·삭제가 빠름** | ④ **C++ `std::map`, Java `TreeMap`, 리눅스 커널** |
| 2-3 | ⑤ **2~3** | 분할/병합 | 회전 불필요 | 교육용 |
| B 트리 | ⑥ **m개** | 분할/병합 | ⑦ **디스크 접근 최소화** | ⑧ **DB 인덱스, 파일 시스템** |

**핵심 질문 답**
1. ⑨ **AVL은 균형 조건이 엄격해서 검색이 빠른 대신 회전이 잦고, 레드·블랙은 느슨해서 검색이 조금 느린 대신 수정이 빠르다.** 실무는 수정도 잦아서 레드·블랙이 표준이 됐어.
2. ⑩ 🔥 **디스크는 블록 단위(예: 4KB)로 읽어서, 키 1개를 읽든 100개를 읽든 비용이 같아.** 그러니 한 노드에 최대한 많이 담아 **높이를 낮추는** 게 유리해.

**실측**
```
데이터 1,000,000개
  이진 트리      : 높이 ≈ 20 → 디스크 접근 20번
  B 트리(자식 100): 높이 ≈ 3  → 디스크 접근 3번  🔥
```
3. ⑪ **안 맞춰줘** ⑫ **균형 트리(레드·블랙 등)나 그 대안**
  - 파이썬은 표준에 균형 트리가 **없어서**: `bisect` + 정렬 리스트, `sortedcontainers` 패키지, 또는 순서가 필요 없으면 `dict`(O(1))

---

## 📌 핵심 3줄 요약

1. **BST는 "왼쪽 < 자신 < 오른쪽"이라는 두 줄짜리 규칙이 전부다.** 단, **자식이 아니라 서브트리 전체**여야 해 — 부모-자식만 보면 (나)·(라) 같은 가짜 BST에 속아. 검증은 **중위 순회가 오름차순인지** 보는 게 제일 간단하고, 그게 `dump()` 가 정렬 출력을 하는 이유이기도 해.
2. **삭제가 세 경우로 갈리는 건 32일차의 "차수" 때문이다.** 자식 0개·1개는 **자식(또는 `None`)을 부모에게 물려주면** 끝이라 한 코드로 합쳐지고, 2개일 때만 **왼쪽 서브트리의 최댓값을 데이터만 복사**해 올려. 그 최댓값은 오른쪽 자식이 없다는 게 보장돼서 경우 C가 재귀하지 않아.
3. **BST의 성능은 삽입 순서가 결정한다.** 정렬된 순서로 넣으면 높이가 `n-1` 이 되어 사실상 연결 리스트야 — 실측에서 검색 비교가 **6번 vs 10,000번**이었지. 21일차 퀵 정렬의 피벗 문제와 정확히 같은 구조고, 해법이 균형 검색 트리(AVL·레드블랙·B트리)야.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 5, 7, 8, 11, 13, 19, 20, 22, 23번)**: 전원 필수
  - **8번 검색 손추적**이 오늘의 기본기. 여기서 "비교하고 한쪽만 본다"를 잡아야 나머지가 읽혀
  - **13번 경우 A**를 확실히 하면 14번 B는 자연스럽게 따라와
- 🟡 **(R-2, 3, 4, 9, 10, 12, 14, 17, 18, 25, 26, 27번)**: 팀 목표선
  - **14번 ⑪⑫(A와 B가 한 코드로 합쳐지는 이유)** 가 중요한 통찰
  - **25번 실측**은 꼭 돌려볼 것 — 6번 vs 10,000번을 눈으로 봐야 균형 트리의 필요성이 와닿아
- 🔴 **(6, 15, 16, 21, 24번)**: 도전
  - **15번 경우 C가 오늘 최고 난도** 🔥🔥 — 세 단계(찾기 → 복사 → 삭제)를 각각 그려볼 것
  - **16번(왜 왼쪽 최댓값인가)** 을 설명할 수 있으면 삭제를 완전히 이해한 거야
  - **6번(가짜 BST 판별)** 은 면접·코테 단골
  - **21번**은 오늘 코드 중 가장 길어. 13~18번 없이는 못 채워
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 27번)

## 🔗 오늘 회수된 개념들

- **32일차 중위 순회** → `dump()` 가 오름차순인 이유 (R-1, 7, 22번)
- **32일차 차수** → 삭제가 세 경우로 갈리는 근거 (PART 5 전체)
- **32일차 높이 / 완전 이진 트리** → 성능의 근거 (4, 25번)
- **32일차 빈 트리** → `None` 이 경우 A·B를 하나로 합침 (14번)
- **32일차 25번 BST 예고** → 오늘 본편 (5, 16번)
- **24일차 힙 정렬** → 완전 이진 트리와 배열 인덱스 (4번)
- **12일차 이진 검색** → BST 검색이 같은 원리 (R-2, 10번)
- **21~22일차 퀵 정렬 피벗** → 삽입 순서가 치우침을 만드는 같은 구조 (26번)
- **29일차 1번 배열 삽입 O(n)** → BST 삽입이 쉬운 이유 (7, 11번)
- **29일차 9번 "삭제 = 화살표 끊기"** → 오늘도 동일 (13번)
- **29일차 11번 `is self.head` 분기** → `if p is self.root` 분기 (17번)
- **31일차 더미가 분기를 없앰** → `None` 이 A·B를 합치는 것과 같은 발상 (14번)

---

> **다음 진도**: 09장이 끝났으니 **10장** 또는 커리큘럼상 다음 단원으로.
>
> 오늘 마지막에 본 **균형 검색 트리**는 이 책 범위를 넘어서지만, 코딩테스트와 실무에서 계속 마주칠 개념이야.
> 파이썬으로 실전 문제를 풀 때는 대부분 `dict`(해시)나 `heapq`(24일차) 로 충분하고, **정렬된 순회가 필요할 때만** 트리를 떠올리면 돼.
>
> 🎉 **오늘 직접 구현하고 테스트까지 돌린 게 진짜 큰 수확이야.** `min_key` 버그를 스스로 발견한 것도 포함해서!